In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.tools import tool
from langchain_core.tools import create_retriever_tool

from langchain_groq import ChatGroq

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from langsmith import Client

from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

from platform import python_version

C:\Users\muham\AppData\Local\Temp\ipykernel_28980\3641853153.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
d:\projects\Langchain module tools and agents\langchain_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
wikipedia_tool = WikipediaQueryRun(api_wrapper = WikipediaAPIWrapper())

In [4]:
vectorstore = Chroma(persist_directory= "./intro-to-ds-lectures",
                     embedding_function= HuggingFaceEmbeddings())

retriever = vectorstore.as_retriever(search_type = 'mmr', 
                                     search_kwargs = {'k':3, 'lambda_mul':0.7})

retriever_tool = create_retriever_tool(retriever = retriever,
                                       name = "Introduction-to-Data-and-Data-Science-Course-Lectures",
                                       description = '''For any questions regarding the Introduction
                                       to Data and Data Science course, you must use this tool.''')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1550.76it/s]
C:\Users\muham\AppData\Local\Temp\ipykernel_28980\1897684529.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory= "./intro-to-ds-lectures",


In [5]:
@tool
def get_python_version() -> str:
    """Useful for questions regarding the version of Python currently used."""
    return python_version()

In [6]:
tools = [wikipedia_tool, retriever_tool, get_python_version]

In [7]:
chat = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [8]:
client = Client()
chat_prompt_template = client.pull_prompt(
    "hwchase17/openai-tools-agent", 
    dangerously_pull_public_prompt=True
)

print(type(chat_prompt_template))

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>


In [9]:
for tool in tools:
    tool.name = tool.name.replace(" ", "_")

print("Sanitized tool names:", [t.name for t in tools])

Sanitized tool names: ['wikipedia', 'Introduction-to-Data-and-Data-Science-Course-Lectures', 'get_python_version']


In [10]:
agent = create_tool_calling_agent(llm = chat,
                                  tools = tools,
                                  prompt = chat_prompt_template)

In [17]:
agent.invoke({"input":"Could you tell me the version of python I'm currently using?",
              "intermediate_steps":response["intermediate_steps"]})

AgentFinish(return_values={'output': 'The current Python version you’re using is **3.13.0**.'}, log='The current Python version you’re using is **3.13.0**.')

In [11]:
agent_executor = AgentExecutor(agent = agent,
                               tools = tools,
                               verbose = True,
                               return_intermediate_steps= True)

In [13]:
response = agent_executor.invoke({"input":"Could you tell me the version of Python I'm currently using?"})



> Entering new AgentExecutor chain...

Invoking: `get_python_version` with `{'': ''}`


3.13.0You’re running **Python 3.13.0**.

> Finished chain.


In [16]:
response

{'input': "Could you tell me the version of Python I'm currently using?",
 'output': 'You’re running **Python\u202f3.13.0**.',
 'intermediate_steps': [(ToolAgentAction(tool='get_python_version', tool_input={'': ''}, log="\nInvoking: `get_python_version` with `{'': ''}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={'reasoning_content': 'We need to call get_python_version.', 'tool_calls': [{'index': 0, 'id': 'fc_9efd7044-a690-465f-9125-52bbee022d24', 'function': {'arguments': '{"":""}', 'name': 'get_python_version'}, 'type': 'function'}]}, response_metadata={'model_provider': 'groq', 'finish_reason': 'tool_calls', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand'}, id='lc_run--01a08285-0f3e-75c1-9638-db6c5d4e7ff0', tool_calls=[{'name': 'get_python_version', 'args': {'': ''}, 'id': 'fc_9efd7044-a690-465f-9125-52bbee022d24', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 257, 'output_to